# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [3]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
        "categories": "../datasets/categories.csv",
        "topics": "../datasets/topics.csv",
        "forms": "../datasets/forms.csv"
    }


# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 categories 테이블

행 수: 17
컬럼: ['category_id', 'category_name', 'category_type', 'parent_category_id', 'category_level', 'category_code', 'description']

첫 5개 행:
   category_id          category_name category_type  parent_category_id  \
0            1  2026 핵심만 담은 노무관리 가이드북          가이드북                 NaN   
1            2              근로조건 서면명시             장                 1.0   
2            3       근로자 명부 및 계약서류 보존             장                 1.0   
3            4          임금 등 각종 금품 지급             장                 1.0   
4            5      근로시간 및 연장근로 한도 위반             장                 1.0   

   category_level category_code                       description  
0               1          G001  고용노동부가 발간한 2026년 노무관리 핵심 가이드북 전체  
1               2           C01       근로계약 체결 및 변경 시 근로조건 서면명시 의무  
2               2           C02               근로자명부, 계약서류 등 보존 의무  
3               2           C03      금품청산, 임금지급, 임금명세서 등 임금 관련 규정  
4               2           C04                 

## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [4]:
for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    df.info()

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()

    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # 고유값 개수
    print("\n[고유값 개수]")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()}개")

    # 카테고리별 데이터 분포
    print("\n[카테고리 분포]")
    for col in df.select_dtypes(include=['object']).columns:
        print(f"\n[{col}]")
        print(df[col].value_counts())


📊 categories 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   category_id         17 non-null     int64  
 1   category_name       17 non-null     str    
 2   category_type       17 non-null     str    
 3   parent_category_id  16 non-null     float64
 4   category_level      17 non-null     int64  
 5   category_code       17 non-null     str    
 6   description         17 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 2.4 KB

[결측치]
parent_category_id    1
dtype: int64

[고유값 개수]
category_id: 17개
category_name: 17개
category_type: 2개
parent_category_id: 1개
category_level: 2개
category_code: 17개
description: 17개

[카테고리 분포]

[category_name]
category_name
2026 핵심만 담은 노무관리 가이드북    1
근로조건 서면명시                1
근로자 명부 및 계약서류 보존         1
임금 등 각종 금품 지급            1
근로시간 및 연장근로 한도 위반        1
휴게시간 부여              

C:\Users\khm35\AppData\Local\Temp\ipykernel_36940\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\khm35\AppData\Local\Temp\ipykernel_36940\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

## 3. Supabase PostgreSQL 연결

In [5]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

C:\Users\khm35\AppData\Local\Temp\ipykernel_36940\1442992373.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['categories', 'departments', 'forms', 'office_floors', 'organizations', 'topics']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [6]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE categories (
	category_id BIGINT, 
	category_name TEXT, 
	category_type TEXT, 
	parent_category_id BIGINT, 
	category_level BIGINT, 
	category_code TEXT, 
	description TEXT
)

/*
3 rows from categories table:
category_id	category_name	category_type	parent_category_id	category_level	category_code	description
1	2026 핵심만 담은 노무관리 가이드북	가이드북	None	1	G001	고용노동부가 발간한 2026년 노무관리 핵심 가이드북 전체
2	근로조건 서면명시	장	1	2	C01	근로계약 체결 및 변경 시 근로조건 서면명시 의무
3	근로자 명부 및 계약서류 보존	장	1	2	C02	근로자명부, 계약서류 등 보존 의무
*/


CREATE TABLE departments (
	dept_id BIGINT, 
	dept_name TEXT, 
	org_id BIGINT, 
	dept_code TEXT, 
	phone TEXT, 
	fax TEXT, 
	floor_location TEXT, 
	description TEXT
)

/*
3 rows from departments table:
dept_id	dept_name	org_id	dept_code	phone	fax	floor_location	description
101	홍보담당관	2	D101	041-521-2080	041-521-2089	본관 8층	홍보업무 총괄 및 보도자료 관리
102	감사관	2	D102	041-521-2040	041-521-2049	본관 4층	감사업무 총괄 및 청렴윤리 관리
103	스마트도시추진과	2	D103	041-521-2205	041-521-2209	본관 7층	스마트도시 정책 및 AI산업 추진
*/


## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [7]:
# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = """
SELECT topic_name, topic_code, page_start, dept_name, dept_phone
FROM topics
WHERE dept_name = '근로기준정책과'
ORDER BY page_start;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT topic_name, topic_code, page_start, dept_name, dept_phone
FROM topics
WHERE dept_name = '근로기준정책과'
ORDER BY page_start;


결과:
[('근로조건 서면명시', '1-1', 6, '근로기준정책과', '044-202-7544'), ('기간제 및 단시간근로자의 근로조건 서면명시', '1-2', 13, '근로기준정책과', '044-202-7544'), ('계약서류 보존', '2-1', 24, '근로기준정책과', '044-202-7544'), ('임금대장', '2-2', 27, '근로기준정책과', '044-202-7534'), ('금품청산', '3-1', 32, '근로기준정책과', '044-202-7529'), ('임금지급', '3-2', 37, '근로기준정책과', '044-202-7529'), ('임금명세서', '3-3', 44, '근로기준정책과', '044-202-7529'), ('도급사업에 대한 임금지급', '3-4', 46, '근로기준정책과', '044-202-7529'), ('휴업수당', '3-5', 48, '근로기준정책과', '044-202-7529'), ('연소자 및 여성의 야간 및 휴일근로', '8-1', 114, '근로기준정책과', '044-202-7555'), ('취업규칙 작성ㆍ신고', '9-1', 146, '근로기준정책과', '044-202-7544'), ('취업규칙 변경', '9-2', 153, '근로기준정책과', '044-202-7544'), ('법령ㆍ단체협약의 준수', '9-3', 159, '근로기준정책과', '044-202-7544'), ('직장 내 괴롭힘 예방', '11-1', 185, '근로기준정책과', '044-202-7539'), ('최저임금의 효력', '12-1', 198, '근로기준정책과', '044-202-7535'), ('최저임금 주지의무', '12-2', 205, '근로기준정책과', '044-202-7555')

In [8]:
# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = """
SELECT c.category_name, t.topic_name, t.dept_name, t.dept_phone
FROM topics t
INNER JOIN categories c ON t.category_id = c.category_id
WHERE c.category_name = '연소자와 모성 보호'
ORDER BY t.topic_code;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT c.category_name, t.topic_name, t.dept_name, t.dept_phone
FROM topics t
INNER JOIN categories c ON t.category_id = c.category_id
WHERE c.category_name = '연소자와 모성 보호'
ORDER BY t.topic_code;


결과:
[('연소자와 모성 보호', '연소자 및 여성의 야간 및 휴일근로', '근로기준정책과', '044-202-7555'), ('연소자와 모성 보호', '산후여성 근로자의 시간외근로', '고용문화개선정책과', '044-202-7471'), ('연소자와 모성 보호', '임산부의 보호', '고용문화개선정책과', '044-202-7471'), ('연소자와 모성 보호', '배우자 출산휴가', '고용문화개선정책과', '044-202-7471'), ('연소자와 모성 보호', '육아휴직', '고용문화개선정책과', '044-202-7475'), ('연소자와 모성 보호', '육아기 근로시간 단축', '고용문화개선정책과', '044-202-7045')]


In [9]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT, SUM, AVG 등 사용

aggregation_query = """
SELECT t.dept_name, COUNT(*) AS topic_count, COUNT(DISTINCT f.form_id) AS form_count
FROM topics t
LEFT JOIN forms f ON f.related_topic_code = t.topic_code
GROUP BY t.dept_name
ORDER BY topic_count DESC;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT t.dept_name, COUNT(*) AS topic_count, COUNT(DISTINCT f.form_id) AS form_count
FROM topics t
LEFT JOIN forms f ON f.related_topic_code = t.topic_code
GROUP BY t.dept_name
ORDER BY topic_count DESC;


결과:
[('근로기준정책과', 18, 10), ('임금근로시간정책과', 10, 6), ('고용문화개선정책과', 9, 2), ('노사협력정책과', 4, 4), ('퇴직연금복지과', 4, 0), ('고용차별개선과', 1, 0)]


## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    system_prompt = f"""
    당신은 고용노동부 "2026 핵심만 담은 노무관리 가이드북" 데이터베이스를 다루는 SQL 전문가입니다.
    사용자의 질문을 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    <데이터베이스 설명>
    - categories: 가이드북의 장(章) 정보 (category_level 1=가이드북 루트, 2=장). parent_category_id로 상위 카테고리 연결
    - topics: 각 장에 속한 절(節) 정보. category_id로 categories 참조. 담당 부서명(dept_name), 직통전화(dept_phone), 부서 위치(dept_location), 관련 법령(related_law), 시작 페이지(page_start) 포함
    - forms: 가이드북에 포함된 서식 목록. related_topic_code 컬럼으로 topics.topic_code를 텍스트로 참조 (진짜 FK 아님, 문자열 매칭 필요)
    </데이터베이스 설명>

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - forms와 topics를 연결할 때는 f.related_topic_code = t.topic_code 로 매칭 (FK 아니므로 JOIN ON에 이 조건 사용)
    - "담당 부서", "전화번호", "위치" 관련 질문은 topics 테이블의 dept_name/dept_phone/dept_location 컬럼을 활용
    - "몇 장", "챕터" 관련 질문은 categories 테이블 활용, 필요 시 topics와 JOIN
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [11]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "근로기준 관련 법령과 담당 부서, 전화번호를 알려주세요."

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: 근로기준 관련 법령과 담당 부서, 전화번호를 알려주세요.


생성된 SQL:
SELECT
    t.topic_name AS 근로기준_관련_주제,
    t.related_law AS 관련_법령,
    t.dept_name AS 담당_부서,
    t.dept_phone AS 전화번호
FROM topics t
WHERE t.related_law LIKE '%근로기준%'
ORDER BY t.topic_id;


실행 결과:
[('근로조건 서면명시', '근로기준법', '근로기준정책과', '044-202-7544'), ('계약서류 보존', '근로기준법', '근로기준정책과', '044-202-7544'), ('임금대장', '근로기준법', '근로기준정책과', '044-202-7534'), ('금품청산', '근로기준법', '근로기준정책과', '044-202-7529'), ('임금지급', '근로기준법', '근로기준정책과', '044-202-7529'), ('임금명세서', '근로기준법', '근로기준정책과', '044-202-7529'), ('도급사업에 대한 임금지급', '근로기준법', '근로기준정책과', '044-202-7529'), ('휴업수당', '근로기준법', '근로기준정책과', '044-202-7529'), ('연장, 야간 및 휴일근로', '근로기준법', '임금근로시간정책과', '044-202-7545'), ('근로시간', '근로기준법', '임금근로시간정책과', '044-202-7545'), ('연장근로의 제한', '근로기준법', '임금근로시간정책과', '044-202-7545'), ('휴게시간 부여', '근로기준법', '임금근로시간정책과', '044-202-7545'), ('유급휴일 부여', '근로기준법', '임금근로시간정책과', '044-202-7973'), ('연차유급휴가 부여', '근로기준법', '임금근로시간정책과', '044-202-7973'), ('연소자 및 여성의 야간 및 휴일근로', '근로기준법', '근로기준정책과', '044-202-7555'

## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [12]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    system_prompt = """
    당신은 고용노동부 "2026 핵심만 담은 노무관리 가이드북" 데이터 분석 전문가입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문(가이드북의 장/절 내용, 담당 부서, 전화번호, 위치, 관련 서식 등)에
    자연스럽게 답변하세요.
    담당 부서 전화번호나 위치가 결과에 포함되어 있다면 정확히 함께 안내하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [13]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "알바생인데 근로계약서 작성과 관련된 법령과 담당 부서, 전화번호를 알려주세요."

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 알바생인데 근로계약서 작성과 관련된 법령과 담당 부서, 전화번호를 알려주세요.


[1] SQL 생성 중...
    SELECT t.topic_name, t.related_law, t.dept_name, t.dept_phone
FROM topics t
WHERE t.topic_name LIKE '%근로계약서%' OR t.description LIKE '%근로계약서%' OR t.topic_name LIKE '%근로조건 서면명시%'
ORDER BY t.page_start;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...


답변:


알바생의 **근로계약서 작성**과 관련해 확인할 수 있는 항목은 다음과 같습니다.

### 관련 법령
1. **근로조건 서면명시**  
   - 관련 법령: **근로기준법**

2. **기간제 및 단시간근로자의 근로조건 서면명시**  
   - 관련 법령: **기간제법**

3. **계약서류 보존**  
   - 관련 법령: **근로기준법**

### 담당 부서
- **근로기준정책과**

### 전화번호
- **044-202-7544**

원하시면 제가 이어서 **알바생 근로계약서에 꼭 들어가야 하는 내용**도 정리해드릴게요.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [ ]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "임금명세서에는 어떤 항목이 반드시 들어가야 하나요?",
    "회사가 퇴사할 때 월급을 언제까지 줘야 하나요?",
    "야간이나 휴일에 일하면 수당을 얼마나 더 받아야 하나요?",
    "연차휴가는 언제부터 며칠이 생기나요?",
    "근무 중에 휴게시간을 꼭 줘야 하나요? 몇 시간 일하면 몇 분 쉬어야 하나요?",
    "육아휴직은 어떤 조건에서 쓸 수 있고, 누구한테 물어보면 되나요?",
    "회사가 최저임금보다 적게 주면 어디에 신고해야 하나요?",
    "직장 내 괴롭힘을 당했을 때 어느 부서에 연락하면 되나요?",
    "기간제나 단시간 근로자도 정규직과 차별받으면 안 되는 건가요? 담당 부서 전화번호도 알려주세요",
    "퇴직금은 얼마나 일해야 받을 수 있고, 담당 부서 위치는 어디인가요?",
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: YOUR_QUESTION_1_HERE

[1] SQL 생성 중...
    SELECT * FROM topics LIMIT 10;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


질문이 `YOUR_QUESTION_1_HERE`로 표시되어 있어, 현재는 어떤 주제를 묻는지 정확히 확인할 수 없습니다.  

다만 실행 결과로 확인되는 주요 항목은 아래와 같습니다.

- **근로조건 서면명시**  
  - 소관 법령: 근로기준법  
  - 담당 부서: **근로기준정책과**  
  - 전화번호: **044-202-7544**  
  - 위치: **정부세종청사 11동 7층**  
  - 내용: 근로계약 체결 시 근로조건을 서면으로 명시하고 교부

- **기간제 및 단시간근로자의 근로조건 서면명시**  
  - 소관 법령: 기간제법  
  - 담당 부서: **근로기준정책과**  
  - 전화번호: **044-202-7544**  
  - 위치: **정부세종청사 11동 7층**  
  - 내용: 기간제·단시간근로자에 대한 근로조건 서면명시 특례

원하시면 다음처럼 구체적으로 질문해 주세요.

- “근로조건 서면명시가 뭐야?”
- “임금지급 담당 부서와 전화번호 알려줘”
- “연장·야간·휴일근로 관련 내용이 뭐야?”
- “임금명세서 서식이나 담당 부서는 어디야?”

질문을 주시면 해당 **장/절 내용, 담당 부서, 전화번호, 위치, 관련 설명**까지 정확히 안내해드리겠습니다.


질문: YOUR_QUESTION_2_HERE

[1] SQL 생성 중...
    SELECT * FROM topics;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


질문에 들어간 내용이 `YOUR_QUESTION_2_HERE`로 표시되어 있어, 어떤 항목을 찾으시는지 정확히 확인되지는 않습니다.

다만, 제공된 `topics` 결과를 기준으로 가이드북에는 다음과 같은 장/절 주제가 포함되어 있습니다:
- **근로조건 서면명시**(1-1) — 근로기준정책과 / 044-202-7544 / 정부세종청사 11동 7층
- **임금지급**(3-2) — 근로기준정책과 / 044-202-7529 / 정부세종청사 11동 7층
- **근로시간**(4-1) — 임금근로시간정책과 / 044-202-7545 / 정부세종청사 11동 6층 624호
- **연차유급휴가 부여**(7-1) — 임금근로시간정책과 / 044-202-7973 / 정부세종청사 11동 6층 624호
- **퇴직금 지급**(10-1) — 퇴직연금복지과 / 044-202-7664 / 정부세종청사 11동 7층 704호
- **직장 내 성희롱 예방**(13-2) — 고용문화개선정책과 / 044-202-7472 / 정부세종청사 11동 5층 508호
- **비정규직 차별 금지**(15-1) — 고용차별개선과 / 044-202-7578 / 정부세종청사 11동 7층 704호

원하시면 제가 바로 아래처럼 정확히 찾아드릴 수 있습니다:
1. **특정 주제명**으로 찾기  
2. **담당 부서/전화번호/위치**만 모아서 보기  
3. **관련 법령별**로 정리하기  
4. **서식/절차 중심**으로 설명하기

원하시는 질문을 예를 들어  
- “임금명세서가 어느 부서 담당인가요?”  
- “육아휴직 관련 내용과 연락처를 알려주세요.”  
처럼 다시 보내주시면 정확히 답변드리겠습니다.


질문: YOUR_QUESTION_3_HERE

[1] SQL 생성 중...
    SELECT t.topic_id, t.topic_name, t.topic_code, t.page_start, t.related_law, t.dept_name, t.dept_phone, t.dept_location, t.description
FROM topics t
ORDER BY t.topic_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


질문 내용이 `YOUR_QUESTION_3_HERE`로 표시되어 있어, 어떤 항목을 찾으시는지 정확히 확인할 수 없습니다.

다만, 쿼리 결과에는 고용노동부 가이드북의 주요 주제별 내용이 정리되어 있습니다. 예를 들어:

- **근로조건 서면명시**  
  - 페이지: 6쪽  
  - 담당 부서: **근로기준정책과**  
  - 전화: **044-202-7544**  
  - 위치: **정부세종청사 11동 7층**
- **임금지급**  
  - 페이지: 37쪽  
  - 담당 부서: **근로기준정책과**  
  - 전화: **044-202-7529**  
  - 위치: **정부세종청사 11동 7층**
- **연장, 야간 및 휴일근로**  
  - 페이지: 55쪽  
  - 담당 부서: **임금근로시간정책과**  
  - 전화: **044-202-7545**  
  - 위치: **정부세종청사 11동 6층 624호**
- **육아휴직**  
  - 페이지: 134쪽  
  - 담당 부서: **고용문화개선정책과**  
  - 전화: **044-202-7475**  
  - 위치: **정부세종청사 11동 5층 508호**
- **퇴직금 지급**  
  - 페이지: 164쪽  
  - 담당 부서: **퇴직연금복지과**  
  - 전화: **044-202-7664**  
  - 위치: **정부세종청사 11동 7층 704호**

원하시면 제가 바로 아래 형식으로 정리해드릴 수 있습니다.

- **장/절명**
- **관련 법령**
- **담당 부서**
- **전화번호**
- **위치**
- **페이지**
- **간단 설명**

찾고 싶은 주제명을 말씀해 주시면 해당 항목만 정확히 답변드리겠습니다.


질문: YOUR_QUESTION_4_HERE

[1] SQL 생성 중...
    SELECT * FROM topics LIMIT 10;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


질문에 들어간 `YOUR_QUESTION_4_HERE`는 실제 질문 내용이 아니어서, 정확히 어떤 항목을 찾으시는지 확인이 어렵습니다.

다만 제공된 SQL 결과에는 아래와 같은 **가이드북 주제 10개**가 포함되어 있습니다.

1. **근로조건 서면명시**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7544**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 근로계약 체결 시 근로조건을 서면으로 명시하고 교부

2. **기간제 및 단시간근로자의 근로조건 서면명시**  
   - 법령: 기간제법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7544**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 기간제ㆍ단시간근로자에 대한 근로조건 서면명시 특례

3. **계약서류 보존**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7544**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 근로계약서 등 계약서류의 보존 의무 및 기간

4. **임금대장**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7534**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 임금대장 작성 및 기재사항

5. **금품청산**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7529**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 퇴직 시 14일 이내 금품청산 의무

6. **임금지급**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7529**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 임금지급의 4대 원칙(통화, 직접, 전액, 정기지급)

7. **임금명세서**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7529**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 임금명세서 교부 의무 및 기재사항

8. **도급사업에 대한 임금지급**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7529**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 도급사업에서의 임금지급 연대책임

9. **휴업수당**  
   - 법령: 근로기준법  
   - 담당부서: 근로기준정책과  
   - 전화번호: **044-202-7529**  
   - 위치: **정부세종청사 11동 7층**  
   - 내용: 사용자 귀책사유에 의한 휴업 시 휴업수당 지급

10. **연장, 야간 및 휴일근로**  
   - 법령: 근로기준법  
   - 담당부서: 임금근로시간정책과  
   - 전화번호: **044-202-7545**  
   - 위치: **정부세종청사 11동 6층 624호**  
   - 내용: 연장ㆍ야간ㆍ휴일근로 가산수당 지급

원하시면 제가 바로 이어서  
- **특정 주제 1개를 골라 상세 설명**하거나  
- **담당 부서/전화번호/위치만 따로 정리**해드릴 수 있습니다.


질문: YOUR_QUESTION_5_HERE

[1] SQL 생성 중...
    SELECT * FROM topics LIMIT 10;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


제공된 SQL 결과는 `topics` 테이블의 일부 항목 10개를 보여주고 있습니다.  
현재 질문이 `YOUR_QUESTION_5_HERE`로 입력되어 있어, **어떤 항목을 문의하신 것인지 특정할 수 없습니다.**

다만 결과에 포함된 주요 항목 예시는 아래와 같습니다.

- **근로조건 서면명시**  
  - 관련 법: 근로기준법  
  - 담당 부서: 근로기준정책과  
  - 전화번호: 044-202-7544  
  - 위치: 정부세종청사 11동 7층  
  - 내용: 근로계약 체결 시 근로조건을 서면으로 명시하고 교부

- **기간제 및 단시간근로자의 근로조건 서면명시**  
  - 관련 법: 기간제법  
  - 담당 부서: 근로기준정책과  
  - 전화번호: 044-202-7544  
  - 위치: 정부세종청사 11동 7층  
  - 내용: 기간제ㆍ단시간근로자에 대한 근로조건 서면명시 특례

- **임금명세서**  
  - 관련 법: 근로기준법  
  - 담당 부서: 근로기준정책과  
  - 전화번호: 044-202-7529  
  - 위치: 정부세종청사 11동 7층  
  - 내용: 임금명세서 교부 의무 및 기재사항

원하시는 주제명이나 장/절 번호를 알려주시면, 해당 항목의 **내용, 담당 부서, 전화번호, 위치, 관련 서식**까지 정리해서 바로 안내드리겠습니다.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용